# 13 Reproducible Research — Reference Solutions

Complete solutions for the reproducible analysis workflow exercises using the Songbai Nursing Home Legionnaires' disease data.

In [ ]:
# Google Colab setup -- skip this cell if running locally
import sys
import os
if 'google.colab' in sys.modules:
    !git clone https://github.com/ancientsky/python4epi.git /content/python4epi 2>/dev/null || true
    os.chdir('/content/python4epi')
    !pip install -q -e .

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

## Question 1: Build an outbreak summary dict

In [ ]:
path = Path("data/synthetic/legionella_outbreak.csv")
df = pd.read_csv(path)
df["infected"] = (df["clinical_severity"] != "not_ill").astype(int)

n_infected = int(df["infected"].sum())
n_deaths = int((df["outcome"] == "dead").sum())

summary = {
    "n_residents": len(df),
    "n_infected": n_infected,
    "n_deaths": n_deaths,
    "attack_rate": f"{df['infected'].mean():.1%}",
    "cfr": f"{n_deaths / n_infected:.1%}",
}

print("=== Outbreak Summary ===")
for k, v in summary.items():
    print(f"  {k}: {v}")

print("\n-> 280 residents, 121 infected, 19 deaths")
print("-> Attack rate 43.2%, CFR 15.7%")

## Question 2: Reproducibility checklist

In [ ]:
checks = {
    "uv.lock exists": Path("uv.lock").exists(),
    "pyproject.toml exists": Path("pyproject.toml").exists(),
    "data file exists": Path("data/synthetic/legionella_outbreak.csv").exists(),
}

print("=== Reproducibility Checklist ===")
for item, ok in checks.items():
    status = "✓" if ok else "✗"
    print(f"  [{status}] {item}")

all_pass = all(checks.values())
print(f"\n-> {'All checks passed!' if all_pass else 'Some checks failed'}")

print("\n=== Why each item matters ===")
print("  uv.lock -> ensures all package versions stay consistent")
print("  pyproject.toml -> defines the project's package requirements")
print("  data file -> no input means no output")

## Question 3 (Challenge): Summary output and verification

In [ ]:
import json
import sys

# Save as CSV
summary_df = pd.DataFrame([summary])
output_path = Path("data/processed")
output_path.mkdir(parents=True, exist_ok=True)
summary_df.to_csv(output_path / "summary.csv", index=False)
print("Saved: data/processed/summary.csv")

# Read back and verify
reloaded = pd.read_csv(output_path / "summary.csv")
print(f"\n=== Verification ===")
print(f"Original n_residents: {summary['n_residents']}")
print(f"Reloaded n_residents: {reloaded['n_residents'].iloc[0]}")
print(f"Match: {summary['n_residents'] == reloaded['n_residents'].iloc[0]}")

# Version info
print(f"\n=== Environment Versions ===")
print(f"  Python: {sys.version.split()[0]}")
print(f"  pandas: {pd.__version__}")
print(f"  numpy: {np.__version__}")

print("\n=== Factors that could cause different results ===")
print("  1. Different package versions (e.g., pandas behavior changes)")
print("  2. Different Python version")
print("  3. Data file modified or missing")
print("  4. Randomness used without a fixed seed")
print("  5. Operating system differences (floating-point precision)")
print("\n-> uv.lock + git solves the first 3 problems")

### Interpretation

- The **summary dict** is the minimal verifiable unit — anyone who runs it should get 280 residents, 121 infected, 19 deaths
- The **checklist** ensures the environment is complete; missing any single item can make the analysis impossible to reproduce
- **Version records** are key to debugging — if results differ, compare versions first
- **The three pillars of reproducibility**: fixed data + version-controlled code + a locked environment